In [28]:
import polars as pl
from rapidfuzz import fuzz

In [29]:
answers = pl.read_parquet("../data_sampled/cases_answer_key.parquet")
corrupted = pl.read_parquet("../data_sampled/corrupted_cases_labeled.parquet")
clean = pl.read_parquet("../data_sampled/cases_sampled.parquet")

In [30]:
clean.lazy().collect_schema()

Schema([('ddl_case_id', String),
        ('year', Int64),
        ('state_code', Int64),
        ('dist_code', Int64),
        ('court_no', Int64),
        ('cino', String),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('type_name', Int64),
        ('purpose_name', Int64),
        ('disp_name', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('date_first_list', Date),
        ('date_last_list', Date),
        ('date_next_list', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('ddl_filing_judge_id', Int64),
        ('ddl_decision_judge_id', Int64),
        ('female_judge_filing', String),
        ('start_date_filing', Date),
        ('end_da

In [31]:
corrupted.lazy().collect_schema()

Schema([('record_id', UInt32),
        ('year', Int64),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('female_judge_filing', String),
        ('female_judge_decision', String),
        ('act_s', List(String)),
        ('section_s', List(String)),
        ('number_sections_ipc', Int64),
        ('criminal', Int64),
        ('bailable_ipc', String),
        ('n_acts_or_sections', UInt32)])

In [32]:
clean = clean.select(["ddl_case_id",
        'year',
        'judge_position',
        'female_defendant',
        'female_petitioner',
        'female_adv_def',
        'female_adv_pet',
        'date_of_filing',
        'date_of_decision', 
        'type_name_s', 
        'disp_name_s', 
        'purpose_name_s', 
        'state_name', 
        'district_name', 
        'court_name', 
        'female_judge_filing', 
        'female_judge_decision', 
        'act_s', 
        'section_s', 
        'number_sections_ipc', 
        'criminal', 
        'bailable_ipc', 
        'n_acts_or_sections'])

In [33]:
print(corrupted.shape)
print(clean.shape)
print(answers.shape)

(50000, 23)
(50000, 23)
(50000, 2)


In [34]:
join = []
for c in corrupted.columns:
    if c not in ["record_id", "act_s", "section_s"]:
        join.append(c)

In [35]:
matches = corrupted.join(clean, on=join, how="inner")
print(matches.height)

4829


In [36]:
matches = corrupted.join(clean, on=join, how="inner", nulls_equal=True)
print(matches.height)

27887


In [37]:
matches.select(pl.col("record_id").n_unique())

record_id
u32
27270


In [38]:
matches_joined = matches.join(answers, on="record_id", how="left", suffix="_true")
pairs = matches_joined.filter(pl.col("ddl_case_id") == pl.col("ddl_case_id_true"))
print(pairs.height)

27255


In [44]:
precision = pairs.height / 27_887
print(f"precision percent of {precision * 100} percent via exact column-column matching.")
correctly_found = pairs.height / 50_000
print(f"recall percent of {correctly_found * 100} percent of original via exactly column-column matching.")

precision percent of 97.73371104815864 percent via exact column-column matching.
recall percent of 54.510000000000005 percent of original via exactly column-column matching.
